# Would behaviour help? How to find out

*The obvious fix for entity resolution is more evidence: what people buy, when they shop,
which devices and addresses they use. This notebook is about how to test whether that fix
works, and about how little public data can settle it.*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omnifroodle/couchbase_notebooks/blob/main/notebooks/experiments/03_would_behaviour_help.ipynb)
[![Open in Codespaces](https://img.shields.io/badge/Open%20in%20Codespaces-2f363d?logo=github&logoColor=white)](https://codespaces.new/omnifroodle/couchbase_notebooks?quickstart=1)

**Experiment.** A method, not a verdict. The data that would settle the question, real people
with two accounts and real households, is not public, so treat every number here as
illustrating the test rather than answering it.
**Claim.** Behaviour can be tested as evidence of identity, but the test that matters is not the
easy one. On real data, what someone buys picks out their own history from four strangers nine
times in ten, and when they shop barely beats chance. In a simulation of households, sharing
even some purchases erodes the separation between housemates while the stranger test goes on
improving, so a test against strangers alone would miss the failure entirely.
**Result.** Products bought: 53.6% found first among 1,000 customers, 90.3% right among five
(chance is 20%). Hour and weekday: 34.1%. Both averaged: 64.0%. Simulated households: beating
the housemate falls from 94.3% to 79.5% at a fifth of baskets shared and 65.5% at half, while
the stranger test rises to 92.7%.
**Requires.** nothing
**Read** ~14 min · **Run** under a minute · **Cost** nothing: no cluster, no model, no API key

[Is this the same customer?](02_is_this_the_same_customer.ipynb) ended on two failures that no
judge could fix from the records alone:

- **A son with his father's name, and no date of birth.** Same name, often the same address and
  phone. Nothing in the record separates them.
- **A returning customer who changed everything.** New surname, new address, new email. Nothing
  left to match on.

The tempting answer is to add evidence: IP addresses, devices, sessions, clicks, purchase history.
Some of it should help. Some of it should make things worse. The only way to know is to test each
kind, and that needs data most teams have and almost nobody publishes.

In [ ]:
# Setup: load the helpers, install anything missing, check what this notebook needs.
import os
import pathlib
import subprocess
import sys

# Cloned on Colab, where there is no local checkout. Override to test a fork.
REPO_URL = os.environ.get("CBNB_REPO_URL", "https://github.com/omnifroodle/couchbase_notebooks")

try:
    import cbnb
except ModuleNotFoundError:
    here = pathlib.Path.cwd()
    root = next((p for p in [here, *here.parents] if (p / "cbnb" / "__init__.py").exists()), None)
    if root is None:
        # Colab: clone the repo so the committed datasets come with it.
        subprocess.check_call(["git", "clone", "--depth", "1", "--quiet", REPO_URL, "cbnb-repo"])
        root = pathlib.Path("cbnb-repo").resolve()
    sys.path.insert(0, str(root))
    import cbnb

settings = cbnb.bootstrap(requires=[])

## 1. Three kinds of evidence, three hypotheses

Behavioural evidence is not one thing. Sorted by who it belongs to, it falls into three kinds,
and each should affect the two failures differently:

| Kind | Examples | Belongs to | Expected effect on the son and his father | Expected effect on the customer who changed everything |
| --- | --- | --- | --- | --- |
| **Stable keys** | payment-card token, device ID, loyalty card, login cookie | one person, usually | Helps, unless they share a card or a computer | Helps most: these survive a move and a new email |
| **Personal behaviour** | what they buy, price range, when they shop, session length, click paths | one person, noisily | Helps, if they shop differently | Helps, if their habits outlast the change |
| **Household-shared** | home IP address, a shared family computer, the delivery address | everyone in the home | **Hurts**: it makes them look more alike | Hurts: a move changes it |

These are hypotheses, and a test must be able to reject them. Two properties of any signal can be
measured before it goes anywhere near a matcher:

- **Distinctiveness.** Among people who are *not* this person, how unusual is this person's
  signal? A signal everyone shares separates nobody.
- **Stability.** Does the signal survive time? A second account is opened later, sometimes much
  later, so it has to be recognisable after a gap.

Both can be measured on real data. What real public data cannot show is the household case,
distinctiveness among people who share a home, and that is exactly the failure that matters.

## 2. Real purchase histories

[Online Retail II](https://archive.ics.uci.edu/dataset/502/online+retail+ii) is two years of real
invoices from a UK online gift shop (CC BY 4.0). The committed sample is 1,000 customers with at
least four invoices each: every invoice they placed, and the products on it. Many of this shop's
customers are small businesses reordering stock, and a business reorders more predictably than a
person. Keep that in mind: it flatters every number below.

It has no names, no addresses and no devices. It has the one kind of behaviour that every shop
keeps: who bought what, and when.

In [2]:
# Load the purchase histories and summarise them.
import numpy as np
import pandas as pd

from cbnb.datasets import ONLINE_RETAIL_CITATION, load_online_retail

retail = load_online_retail()
baskets = retail.baskets
per_customer = baskets.groupby("customer_id").size()
print(ONLINE_RETAIL_CITATION)
print(f"{baskets.customer_id.nunique():,} customers, {len(baskets):,} invoices, "
      f"{len({code for items in baskets['items'] for code in items}):,} distinct products, "
      f"{baskets.date.min():%b %Y} to {baskets.date.max():%b %Y}")
print(f"invoices per customer: median {per_customer.median():.0f}, "
      f"quartiles {per_customer.quantile(0.25):.0f}–{per_customer.quantile(0.75):.0f}")

Chen, D. (2019). Online Retail II [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5CG6D (CC BY 4.0).
1,000 customers, 12,045 invoices, 4,401 distinct products, Dec 2009 to Dec 2011
invoices per customer: median 7, quartiles 5–12


In [3]:
# One customer's first three invoices, with product names.
names = retail.products.set_index("stock_code").description
example = baskets[baskets.customer_id == baskets.customer_id.iloc[0]].head(3)
for invoice in example.itertuples():
    shown = ", ".join(names.get(code, code).lower() for code in invoice.items[:6])
    print(f"{invoice.date:%d %b %Y %H:%M} · {len(invoice.items)} products: {shown}, …")

31 Oct 2010 14:20 · 40 products: jumbo bag woodland animals, party food shopper bag, woodland charlotte bag, sandwich bath sponge, heart calculator, pink poodle hanging decoration, …
07 Dec 2010 14:57 · 31 products: black ear muff headphones, camouflage ear muff headphones, boom box speaker boys, bathroom metal sign, red toadstool led night light, large heart measuring spoons, …
26 Jan 2011 14:30 · 29 products: woodland charlotte bag, sandwich bath sponge, set/2 red spotty tea towels, red retrospot oven glove double, red retrospot oven glove, red toadstool led night light, …


## 3. The test: split one customer in two

No public dataset links one real person's two accounts, so make the link: take each customer's
invoices in date order, call the first half one account and the second half another, and forget
that they are the same person. Then ask whether the later account's behaviour finds the earlier
one among everybody's.

This is the standard way to test behavioural linking without labels, and its answer is real:
real people, real habits, real drift over time. It measures distinctiveness among strangers and
stability over time, and nothing else.

In [4]:
# Split every customer's invoices in date order: the first half is one account, the second half another.
baskets = baskets.sort_values(["customer_id", "date"]).reset_index(drop=True)
position = baskets.groupby("customer_id").cumcount()
baskets["account"] = np.where(position < baskets.groupby("customer_id").invoice.transform("size") / 2,
                              "earlier", "later")
customers = sorted(baskets.customer_id.unique())
row = {cid: i for i, cid in enumerate(customers)}
baskets.groupby("account").agg(invoices=("invoice", "size"), customers=("customer_id", "nunique"))

,invoices,customers
account,,
earlier,6238,1000
later,5807,1000


Each account becomes a vector over products: how often it bought each one, weighted so that
products everybody buys count for less than products few people buy (TF-IDF, the same weighting
a keyword search uses). Two accounts' **similarity** is the cosine between their vectors.

In [5]:
# A vector per account over the products it bought, weighted by how rare each product is.
codes = sorted({code for items in baskets["items"] for code in items})
column = {code: j for j, code in enumerate(codes)}


def product_counts(account):
    counts = np.zeros((len(customers), len(codes)), dtype=np.float32)
    for cid, items in baskets.loc[baskets.account == account, ["customer_id", "items"]].itertuples(index=False):
        counts[row[cid], [column[code] for code in items]] += 1
    return counts


def unit(vectors):
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    return vectors / np.where(norms == 0, 1, norms)


earlier, later = product_counts("earlier"), product_counts("later")
buyers = ((earlier + later) > 0).sum(axis=0)
rarity = np.log(2 * len(customers) / (1 + buyers))          # inverse document frequency
by_products = unit(later * rarity) @ unit(earlier * rarity).T  # row: a later account; column: an earlier one

Two numbers per signal. **Found first among everyone**: the later account's most similar earlier
account is its own, out of 1,000. **Right among five**: the true earlier account beats four
strangers drawn at random. Five mirrors the candidate lists in `experiments/02`, whose candidates
came from a name search and are unrelated to what anyone buys. By chance these would be 0.1% and
20%.

In [6]:
# Score a signal: how often each later account finds its own earlier account.
rng = np.random.default_rng(0)
strangers = np.array([rng.choice(np.delete(np.arange(len(customers)), i), 4, replace=False)
                      for i in range(len(customers))])


def score(similarity):
    own = similarity.diagonal()
    outranked_by = (similarity > own[:, None]).sum(axis=1)   # ties count in the account's favour
    return pd.Series({
        "found first among everyone": (outranked_by == 0).mean(),
        "right among five": (own > np.take_along_axis(similarity, strangers, axis=1).max(axis=1)).mean(),
        "median rank": np.median(outranked_by) + 1,
        "similarity to own account": np.median(own),
        "similarity to a stranger": np.median(similarity[~np.eye(len(own), dtype=bool)]),
    })


score(by_products).to_frame("products bought").T.style.format(
    {"found first among everyone": "{:.1%}", "right among five": "{:.1%}", "median rank": "{:.0f}",
     "similarity to own account": "{:.2f}", "similarity to a stranger": "{:.2f}"})

,found first among everyone,right among five,median rank,similarity to own account,similarity to a stranger
products bought,53.6%,90.3%,1,0.24,0.02


Half of the later accounts find their own earlier account first out of a thousand, and nine in
ten beat four strangers. The last two columns say why: a customer's two halves share products
(median similarity 0.24), while two strangers' purchases barely overlap (0.02). Among strangers,
what people buy is a strong signal. Remember the shop, though: businesses reordering the same
stock make this easier than it would be for people.

## 4. Not all behaviour is evidence

The same test for *when* people shop: a vector per account over hour of day and day of week. It
is the kind of signal that looks like a fingerprint on a dashboard. Then both signals together,
with their similarities averaged.

In [7]:
# The same test for when people shop, and for both signals together.
def shopping_times(account):
    times = np.zeros((len(customers), 24 + 7), dtype=np.float32)
    for cid, when in baskets.loc[baskets.account == account, ["customer_id", "date"]].itertuples(index=False):
        times[row[cid], when.hour] += 1
        times[row[cid], 24 + when.weekday()] += 1
    return unit(times)


by_time = shopping_times("later") @ shopping_times("earlier").T
signals = pd.DataFrame({"products bought": score(by_products),
                        "hour and weekday": score(by_time),
                        "both, averaged": score((by_products + by_time) / 2)}).T
signals.style.format({"found first among everyone": "{:.1%}", "right among five": "{:.1%}",
                      "median rank": "{:.0f}", "similarity to own account": "{:.2f}",
                      "similarity to a stranger": "{:.2f}"})

,found first among everyone,right among five,median rank,similarity to own account,similarity to a stranger
products bought,53.6%,90.3%,1,0.24,0.02
hour and weekday,1.3%,34.1%,301,0.47,0.37
"both, averaged",24.8%,64.0%,46,0.36,0.20


Shopping times barely beat chance: 34.1% right among five, against 20% by guessing. The last two
columns show why. Any two customers' shopping times look alike, with a median similarity of 0.37
between strangers against 0.47 for the same person, so there is little to tell them apart.
Averaged with products, that noise drowns the product signal: 64.0% right among five, much worse
than products alone.

A signal can sound personal and still separate nobody, and adding it to a good one can make the
result worse. Test each signal on its own before combining anything, and when combining, weight
by what each signal was measured to be worth, not equally.

## 5. How much history, and how long ago

A second account rarely arrives with half of someone's history behind it. The same score, split
by how many invoices the customer placed in all, and by the gap between the last invoice of the
earlier account and the first of the later one.

In [8]:
# Right-among-five for products bought, by length of history and by the gap between the two accounts.
own = by_products.diagonal()
right = pd.Series(own > np.take_along_axis(by_products, strangers, axis=1).max(axis=1), index=customers)
invoices = baskets.groupby("customer_id").size().reindex(customers)
ends = baskets.groupby(["customer_id", "account"]).date.agg(["min", "max"]).unstack()
gap_days = (ends[("min", "later")] - ends[("max", "earlier")]).dt.days.reindex(customers)

by_history = right.groupby(pd.cut(invoices, [3, 5, 9, 20, np.inf],
                                  labels=["4–5 invoices", "6–9", "10–20", "over 20"]), observed=True)
by_gap = right.groupby(pd.cut(gap_days, [-1, 7, 30, 90, np.inf],
                              labels=["under a week", "1 week – 1 month", "1–3 months", "over 3 months"]),
                       observed=True)
pd.concat({"history": by_history.agg(["mean", "size"]), "gap": by_gap.agg(["mean", "size"])}).rename(
    columns={"mean": "right among five", "size": "customers"}).style.format({"right among five": "{:.1%}"})

History matters more than time. With only four or five invoices, the right account wins 80.1%
of the time; with more than twenty, 99.0%. A gap of over three months costs about six points
(85.1%, against 90.9% within a week).

Plan around the thin rows. A record arriving at sign-up has no purchase history at all, so
behaviour can only help link an account after it has been used for a while.

## 6. The household dial: a simulation

> **Everything in this section is made up.** No public data has households, so this section
> builds them, and its numbers describe the simulation, not the world. It shows the *shape* of
> the problem and how a test should be set up, and it is the part to replace with your own
> labelled households.

Pair the 1,000 real customers at random into two-person households. Then turn a dial: each
basket either person bought also shows up, with probability *overlap*, in their housemate's
account, as it would when one person orders for the home under both accounts. Each person keeps
their own real purchases. At every setting, ask two questions of every later account: does it
still find its own earlier account ahead of **its housemate's**, and still ahead of four
**strangers**?

In [9]:
# Pair the customers at random into two-person households.
households = np.random.default_rng(1).permutation(len(customers)).reshape(-1, 2)
housemate = np.empty(len(customers), dtype=int)
housemate[households[:, 0]], housemate[households[:, 1]] = households[:, 1], households[:, 0]
print(f"{len(households)} households; e.g. customers {customers[households[0, 0]]} and {customers[households[0, 1]]}")

500 households; e.g. customers 16668 and 16320


In [10]:
# Sweep the dial: at each overlap, share baskets between housemates, then score both tests.
def with_shared(account, overlap, seed):
    counts = product_counts(account)
    coin = np.random.default_rng(seed)
    for cid, items in baskets.loc[baskets.account == account, ["customer_id", "items"]].itertuples(index=False):
        if coin.random() < overlap:
            counts[housemate[row[cid]], [column[code] for code in items]] += 1
    return counts


dial = {}
for overlap in [0.0, 0.1, 0.2, 0.3, 0.5, 0.7, 0.9]:
    similarity = unit(with_shared("later", overlap, 2) * rarity) @ unit(with_shared("earlier", overlap, 3) * rarity).T
    own = similarity.diagonal()
    dial[overlap] = {
        "beats the housemate": (own > similarity[np.arange(len(own)), housemate]).mean(),
        "right among five strangers": (own > np.take_along_axis(similarity, strangers, axis=1).max(axis=1)).mean(),
    }
pd.DataFrame(dial).T.rename_axis("share of baskets shared").style.format("{:.1%}").format_index("{:.0%}")

,beats the housemate,right among five strangers
share of baskets shared,,
0%,94.3%,90.3%
10%,86.9%,90.1%
20%,79.5%,90.1%
30%,73.8%,91.0%
50%,65.5%,92.7%
70%,53.4%,94.3%
90%,33.3%,95.8%


Read the two columns against each other. With nothing shared, a later account beats its
housemate 94.3% of the time: two random customers are as different as any two strangers. Share a
tenth of the baskets and that falls to 86.9%, a fifth to 79.5%, a half to 65.5%, against a coin
flip's 50%. At 90% it drops to 33.3%, below chance: whoever buys more takes over both accounts,
so the lighter buyer's later account looks more like the heavier buyer's history than their own.

Meanwhile the strangers column *rises*, from 90.3% to 95.8%, because shared baskets give every
account more to be recognised by. **A test run only against strangers reports behaviour
improving at exactly the moment it stops separating the people who matter.** That is the reason
to test within households separately, and the reason this notebook cannot say how behaviour
does for real ones: nobody knows where real households sit on this dial without labelled
households to measure.

## 7. What this shows, and what it doesn't

**It shows** that what someone buys can be distinctive among strangers, and that it fades with
time and with thin histories. It shows that a signal which sounds personal, when they shop, can be
close to useless, and that adding it to a good one dilutes it. Both are reasons to test each
signal on its own before combining anything.

**It does not show** that behaviour separates a son from his father. Everyone in this test is a
stranger to everyone else. People who share a home share a delivery address, often a card, and
often a taste, because they buy for the same house. Nothing here measures that overlap, and no
public dataset found so far can (see [`docs/datasets.md`](../../docs/datasets.md)); section 6
only shows how quickly it could go wrong.

**It also leans on this shop.** Businesses reordering stock are more predictable than people, and
the split assumes a second account behaves like the first. Often a second account exists
*because* something changed: a new device, a gift, a move.

## 8. Running the real test on your own data

Everything above can be run on your own records, and the parts this notebook cannot show only
can be.

1. **Labels first.** The ground truth is accounts your support team has already merged, and
   pairs a person has reviewed and ruled on. Keep the "no" decisions too: a household that was
   *not* merged is the most valuable label you have.
2. **Measure each signal alone.** Distinctiveness among strangers, as in section 3; stability
   over the gaps your second accounts actually show, as in section 5; and, new with your labels,
   **distinctiveness within households**: section 6's test, with real housemates, meaning the
   other people at the same address. Report it separately. Averaged in with strangers it
   disappears, and section 6 shows it moving the opposite way.
3. **Then add them one at a time** to the matcher from `experiments/02`: identity fields, then
   stable keys, then behaviour, then household-shared signals. Judge each step by what it does to
   the automatic share at a fixed error rate, broken down by the kind of record, because an
   average will hide a signal that fixes strangers and breaks households.
4. **Give the model a summary, and code the comparisons.** A judge reads "mostly Christmas
   decorations and tea sets, weekday mornings" better than a vector, and code should compute
   "3 products in common" before the model is asked, not leave it to the model.
5. **Check you may use it.** IP addresses and device fingerprints are personal data under GDPR
   and CCPA. Using them to connect people needs a lawful basis and usually a notice, and sending
   them to a third-party model is a data transfer. Summarising behaviour locally and sending only
   the summary avoids most of that.

> **On Couchbase** — behaviour is another view on the same profile document: a purchase vector
> beside the identity fields and the resolved customer id, kept current as orders arrive. A
> vector index on it finds behavioural candidates the name search misses, which is the search
> this notebook's split test is standing in for.

## Where to take this

- **Taste, not just baskets.** The dial only models shared orders. Real housemates also like
  similar things. A second dial, pairing customers by similar taste instead of at random, would
  show how much faster the separation fails.
- **A second shop.** The [OTTO dataset](https://github.com/otto-de/recsys-dataset) (CC BY 4.0) is
  clicks and orders from a consumer retailer, closer to people than a wholesaler's reorders, and
  large enough to need a download step.
- **Behaviour as a candidate search.** Run the split test through a Couchbase vector index, and
  measure what it adds to the name search's 89% reach in `experiments/02`.